This notebook attempts to showcase the creation of a suitable list of largest mids (machine identifier) that are contained in both audio datasets Audioset and FSD50k. This list of mids is needed to produce a subset of needed segment files (less files needed means less conversions, I/O, and storage) and to produce accurate target labels for each file. 

In [1]:
import pandas as pd

In [2]:
# Read in the metafiles of both fsd50k and audioset

# Change column names to be more cohesive with audioset
fsd50k_class_vocab = pd.read_csv(r'C:\Users\mp431591\Documents\FSD50k_label_files\vocabulary.csv',
                                 header=None, names=["display_name", "mids"])

fsd50k_mids = set(fsd50k_class_vocab["mids"])

# Same solumn name change as for fsd50k
audioset_mid_to_label_mapping = pd.read_csv(r'C:\Users\mp431591\Documents\audioset_label_files\mid_to_display_name.tsv',
                                     sep="\t", header=None, names=["mids", "display_name"])
audioset_mid_to_label_mapping.head()

,mids,display_name
0,/g/11b630rrvh,Kettle whistle
1,/g/122z_qxw,Firecracker
2,/m/01280g,Wild animals
3,/m/012f08,Motor vehicle (road)
4,/m/012n7d,Ambulance (siren)


In [3]:
# Read in audiosets strongly annotated training split meta file to quantify largest classes.
# While the evaluation splits information could also be used, the training split
# is larger and is most likely good enough.

# Audioset train labels
audioset_train_classes = pd.read_csv(r'C:\Users\mp431591\Documents\audioset_label_files\audioset_train_strong.tsv',
                                     sep="\t")
audioset_train_classes.head()

,segment_id,start_time_seconds,end_time_seconds,label
0,b0RFKhbpFJA_30000,0.000,10.000,/m/03m9d0z
1,b0RFKhbpFJA_30000,4.753,5.720,/m/05zppz
2,b0RFKhbpFJA_30000,0.000,10.000,/m/07pjwq1
3,b0RFKhbpFJA_30000,6.899,7.010,/m/07qjznt
4,b0RFKhbpFJA_30000,8.534,9.156,/t/dd00092


In [4]:
# Drop temporal information since we're aiming for weak labels
audioset_train_classes = audioset_train_classes.loc[:, ['segment_id', 'label']]
audioset_train_classes.head(5)

,segment_id,label
0,b0RFKhbpFJA_30000,/m/03m9d0z
1,b0RFKhbpFJA_30000,/m/05zppz
2,b0RFKhbpFJA_30000,/m/07pjwq1
3,b0RFKhbpFJA_30000,/m/07qjznt
4,b0RFKhbpFJA_30000,/t/dd00092


In [5]:
# Rename labels column to mids column for more accurate meaning and better coherence with FSD50k
audioset_train_classes.rename(columns={'label': 'mids'}, inplace=True)
audioset_train_classes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 934821 entries, 0 to 934820
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   segment_id  934821 non-null  object
 1   mids        934821 non-null  object
dtypes: object(2)
memory usage: 14.3+ MB


In [6]:
# Drop duplicates in case any 10 sec clip has multiple instances of the same sound
# A quick manual look confirms these exist for at least audioset train
audioset_train_no_dups = audioset_train_classes.drop_duplicates()
audioset_train_no_dups.info()

<class 'pandas.core.frame.DataFrame'>
Index: 432296 entries, 0 to 934813
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   segment_id  432296 non-null  object
 1   mids        432296 non-null  object
dtypes: object(2)
memory usage: 9.9+ MB


In [7]:
# Drop rows where the mid is not found in the fsd50k mids.
rows_w_good_mid = audioset_train_no_dups.mids.isin(fsd50k_mids)
rows_w_good_mid
audioset_good_rows = audioset_train_no_dups[rows_w_good_mid]
audioset_good_rows.info()

<class 'pandas.core.frame.DataFrame'>
Index: 277487 entries, 0 to 934813
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   segment_id  277487 non-null  object
 1   mids        277487 non-null  object
dtypes: object(2)
memory usage: 6.4+ MB


In [8]:
# Count mid frequencies
audioset_train_labels_by_mid_count = (audioset_good_rows.value_counts() # add a value to unique rows (in practice should be 1)
                     .groupby(by='mids') # organize by mid
                     .sum() # Sum the mid specific value i.e, in how many segments does the mid appear
                     .sort_values(ascending=False))
audioset_train_labels_by_mid_count

mids
/m/05zppz     35107
/m/04rlf      27135
/t/dd00077    26219
/m/07qjznt    18705
/m/0lyf6      16523
              ...  
/m/01dwxx         8
/t/dd00012        8
/m/07q8k13        4
/m/09hlz4         4
/m/0ch8v          3
Name: count, Length: 165, dtype: int64

In [9]:
# Get the top 50 mids that can be used to narrow down the necessary files.
top_50_mids = audioset_train_labels_by_mid_count[0:50]
top_50_mids = set(top_50_mids.keys())

def audioset_mid_to_display_name(mid):
    return audioset_mid_to_label_mapping[audioset_mid_to_label_mapping['mids'] == mid]['display_name'].iloc[0]

for mid in top_50_mids:
    print(audioset_mid_to_display_name(mid))

Vehicle horn, car horn, honking, toot
Motor vehicle (road)
Explosion
Walk, footsteps
Crumpling, crinkling
Traffic noise, roadway noise
Mechanisms
Bird vocalization, bird call, bird song
Whispering
Cough
Tick
Human voice
Car
Female speech, woman speaking
Crowd
Conversation
Shout
Chirp, tweet
Cricket
Gasp
Child speech, kid speaking
Tap
Water
Music
Speech
Cheering
Insect
Giggle
Applause
Breathing
Dog
Bark
Accelerating, revving, vroom
Bell
Wind
Singing
Splash, splatter
Laughter
Gunshot, gunfire
Speech synthesizer
Male singing
Male speech, man speaking
Train
Screaming
Female singing
Truck
Clapping
Thump, thud
Animal
Alarm


In [11]:
# Save the object as a pickle object for further downstream use.
from utility_funcs import pickle_save
pickle_save("top_50_mids.pckl", top_50_mids)